# Engineering Productivity & Cloud Cost Intelligence — EDA

This notebook explores the fictional synthetic engineering dataset that ships with this project. It is not Zepto data or a representation of any actual company. The purpose is to demonstrate a repeatable Business Analyst workflow: validate inputs, understand distributions, calculate KPIs, identify patterns and state limitations.

**Run from the project root** (or let the path helper below locate it). Install packages first with `pip install -r requirements.txt`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw").exists():
    raise FileNotFoundError("Open this notebook from the project folder so data/raw can be found.")
sys.path.insert(0, str(ROOT / "src"))
from analytics import load_data, run_all_analyses, data_as_of_date
from validate_data import validate

data = load_data(ROOT / "data" / "raw")
print("Project root:", ROOT)
print("Dataset as-of date:", data_as_of_date(data).date())

Project root: /mnt/data/Engineering_Productivity_Cloud_Cost_Analytics
Dataset as-of date: 2026-11-01


## 1. Table inventory and basic validation
Check row counts, table grain and key integrity before analysing metrics.

In [2]:
inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns), "missing_cells": int(frame.isna().sum().sum())}
    for name, frame in data.items()
]).sort_values("table")
inventory

,table,rows,columns,missing_cells
6,capacity,910,10,0
5,cloud_costs,19110,8,0
4,deployments,684,11,0
1,engineers,50,7,0
7,incidents,148,12,127
2,sprints,91,7,0
3,tasks,939,15,407
0,teams,7,7,0


In [3]:
validation_report = validate(data)
display(validation_report)
errors = validation_report[(validation_report.status == "FAIL") & (validation_report.severity == "ERROR")]
assert errors.empty, "Resolve validation errors before proceeding."

,check,status,severity,detail
0,teams primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
1,engineers primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
2,sprints primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
3,tasks primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
4,deployments primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
5,cloud_costs primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
6,incidents primary key uniqueness,PASS,ERROR,Duplicate key rows: 0
7,capacity composite key uniqueness,PASS,ERROR,Duplicate date/team rows: 0
8,engineers.team_id foreign key,PASS,ERROR,Orphan team references: 0
9,sprints.team_id foreign key,PASS,ERROR,Orphan team references: 0


## 2. Understand the operating footprint
The teams intentionally have different sizes. Use headcount and workload context when comparing them.

In [4]:
team_profile = (data["teams"][["team_id", "team_name", "headcount", "monthly_cloud_budget_inr"]]
    .merge(data["engineers"].groupby("team_id").size().rename("engineer_records"), on="team_id", how="left")
    .sort_values("headcount", ascending=False))
team_profile

,team_id,team_name,headcount,monthly_cloud_budget_inr,engineer_records
2,T03,Fulfillment Systems,10,285000,10
1,T02,Core Platform,9,335000,9
0,T01,Checkout & Payments,8,245000,8
3,T04,Consumer Mobile,7,165000,7
4,T05,Data & Analytics,6,195000,6
5,T06,SRE & Infrastructure,5,310000,5
6,T07,Merchant Integrations,5,125000,5


In [5]:
fig = px.bar(team_profile, x="team_name", y="headcount", title="Fictional engineering team sizes", labels={"team_name":"Team", "headcount":"Headcount"})
fig.update_layout(xaxis_tickangle=-25)
fig.show()

## 3. Task and sprint productivity
Sprint completion is calculated using completed story points for tasks marked as committed divided by all committed story points. Story points are team-relative, so compare trends within teams rather than treating points as a standardized cross-team measure.

In [6]:
analysis = run_all_analyses(data)
sprint_kpis = analysis["sprint_kpis_by_team_sprint"]
team_tasks = analysis["team_task_summary"]
display(team_tasks.sort_values("commit_completion_pct", ascending=False).round(2))

,team_id,task_count,done_tasks,blocked_tasks,open_tasks,committed_points,completed_committed_points,mean_cycle_time_days,median_cycle_time_days,mean_blocked_days,task_completion_pct,commit_completion_pct,team_name,headcount
1,T02,175,152,3,23,564,471,5.44,4.85,0.31,86.86,83.51,Core Platform,9
5,T06,104,85,4,19,305,253,5.00,5.46,0.48,81.73,82.95,SRE & Infrastructure,5
4,T05,129,107,6,22,400,328,4.51,3.75,0.43,82.95,82.00,Data & Analytics,6
3,T04,110,81,6,29,349,261,4.46,3.50,0.72,73.64,74.79,Consumer Mobile,7
0,T01,139,110,9,29,382,283,4.75,4.04,0.67,79.14,74.08,Checkout & Payments,8
6,T07,109,80,9,29,358,254,5.26,4.64,0.59,73.39,70.95,Merchant Integrations,5
2,T03,173,131,9,42,569,401,5.26,4.58,0.73,75.72,70.47,Fulfillment Systems,10


In [7]:
fig = px.line(sprint_kpis, x="sprint_number", y="commit_completion_pct", color="team_name", markers=True,
              title="Committed story-point completion by sprint",
              labels={"sprint_number":"Sprint number", "commit_completion_pct":"Completion (%)", "team_name":"Team"})
fig.show()

In [8]:
status_mix = data["tasks"].groupby(["team_id", "status"]).size().reset_index(name="task_count")
status_mix = status_mix.merge(data["teams"][["team_id", "team_name"]], on="team_id")
fig = px.bar(status_mix, x="team_name", y="task_count", color="status", barmode="stack",
             title="Task status mix by team", labels={"team_name":"Team", "task_count":"Tasks"})
fig.update_layout(xaxis_tickangle=-25)
fig.show()

## 4. Deployment reliability
Use Production-only deployments for this view. Inspect event counts alongside failure rates; a small denominator can make percentages volatile.

In [9]:
deployment_summary = analysis["production_deployment_team_summary"].sort_values("failure_rate_pct", ascending=False)
display(deployment_summary.round(2))
fig = px.bar(deployment_summary, x="team_name", y="failure_rate_pct",
             title="Production deployment failure rate by team",
             labels={"team_name":"Team", "failure_rate_pct":"Failure rate (%)"})
fig.update_layout(xaxis_tickangle=-25)
fig.show()

,team_id,production_deployments,successful_deployments,failed_deployments,rollbacks,emergency_deployments,failure_rate_pct,rollback_rate_pct,successful_production_deployments_per_week,team_name,headcount
6,T07,21,18,3,2,0,14.29,9.52,0.70,Merchant Integrations,5
0,T01,67,60,7,6,3,10.45,8.96,2.33,Checkout & Payments,8
2,T03,56,52,4,4,1,7.14,7.14,2.02,Fulfillment Systems,10
4,T05,33,31,2,2,6,6.06,6.06,1.21,Data & Analytics,6
1,T02,90,85,5,5,2,5.56,5.56,3.31,Core Platform,9
3,T04,21,20,1,2,3,4.76,9.52,0.78,Consumer Mobile,7
5,T06,71,69,2,1,2,2.82,1.41,2.68,SRE & Infrastructure,5


## 5. Cloud cost and budget variance
The budget column is a synthetic daily allocation, not an invoice. Separate absolute spend from relative variance.

In [10]:
cloud_month = analysis["cloud_cost_by_team_month"]
display(cloud_month.sort_values(["month", "actual_cost_inr"], ascending=[True, False]).round(2).head(15))
fig = px.line(cloud_month, x="month", y="actual_cost_inr", color="team_name", markers=True,
              title="Monthly simulated cloud spend", labels={"month":"Month", "actual_cost_inr":"Cost (INR)"})
fig.show()

,team_id,month,actual_cost_inr,budget_inr,cost_records,variance_inr,variance_pct,team_name
7,T02,2026-05,240839.05,302580.60,420,-61741.55,-20.40,Core Platform
35,T06,2026-05,212201.24,280000.00,420,-67798.76,-24.21,SRE & Infrastructure
14,T03,2026-05,210365.41,257419.12,420,-47053.71,-18.28,Fulfillment Systems
0,T01,2026-05,168685.02,221290.44,420,-52605.42,-23.77,Checkout & Payments
28,T05,2026-05,137877.75,176129.24,420,-38251.49,-21.72,Data & Analytics
21,T04,2026-05,121191.05,149032.52,420,-27841.47,-18.68,Consumer Mobile
42,T07,2026-05,86326.21,112903.28,420,-26577.07,-23.54,Merchant Integrations
8,T02,2026-06,270871.89,334999.80,450,-64127.91,-19.14,Core Platform
36,T06,2026-06,262209.96,310000.20,450,-47790.24,-15.42,SRE & Infrastructure
15,T03,2026-06,242734.35,285000.00,450,-42265.65,-14.83,Fulfillment Systems


In [11]:
service_cost = analysis["cloud_cost_by_service_environment"].sort_values("actual_cost_inr", ascending=False)
display(service_cost.head(15).round(2))

,team_id,service_name,environment,actual_cost_inr,budget_inr,variance_inr,variance_pct,team_name
19,T02,Kubernetes,Production,427046.88,471333.23,-44286.35,-9.40,Core Platform
82,T06,Kubernetes,Production,381751.79,421620.57,-39868.78,-9.46,SRE & Infrastructure
88,T06,Observability,Production,331072.79,363465.63,-32392.84,-8.91,SRE & Infrastructure
43,T03,Order Services,Production,328073.27,360885.93,-32812.66,-9.09,Fulfillment Systems
16,T02,Compute,Production,325445.71,361354.91,-35909.20,-9.94,Core Platform
13,T01,Payment API,Production,307544.41,344705.76,-37161.35,-10.78,Checkout & Payments
31,T03,Compute,Production,299892.03,320788.30,-20896.27,-6.51,Fulfillment Systems
25,T02,Managed Database,Production,297022.35,329933.14,-32910.79,-9.97,Core Platform
64,T05,Data Warehouse,Production,270875.93,301794.24,-30918.31,-10.24,Data & Analytics
7,T01,Managed Database,Production,259614.66,287255.81,-27641.15,-9.62,Checkout & Payments


## 6. Capacity and resource utilization
Calculate utilization as the ratio of total logged hours to total available hours. A high figure can indicate overtime or lack of buffer; it is not automatically evidence of high productivity.

In [12]:
capacity_summary = analysis["capacity_team_summary"].sort_values("utilization_pct", ascending=False)
display(capacity_summary.round(2))
fig = px.bar(capacity_summary, x="team_name", y="utilization_pct",
             title="Weighted utilization by team", labels={"team_name":"Team", "utilization_pct":"Utilization (%)"})
fig.update_layout(xaxis_tickangle=-25)
fig.show()

,team_id,available_hours,logged_hours,delivery_hours,support_hours,incident_hours,meeting_and_admin_hours,overtime_hours,utilization_pct,team_name,headcount
5,T06,5008.0,4622.76,2578.13,949.26,548.57,546.79,59.44,92.31,SRE & Infrastructure,5
2,T03,10121.5,9121.34,5252.39,1794.00,837.17,1237.76,58.83,90.12,Fulfillment Systems,10
1,T02,9087.5,8061.25,5531.68,1247.09,355.17,927.12,69.83,88.71,Core Platform,9
0,T01,8032.0,6622.03,4571.80,906.22,365.33,778.79,44.53,82.45,Checkout & Payments,8
4,T05,5972.5,4865.45,3304.61,652.13,280.80,627.97,12.55,81.46,Data & Analytics,6
3,T04,7071.5,5541.48,3709.45,836.03,282.79,713.24,5.91,78.36,Consumer Mobile,7
6,T07,5059.5,3757.13,2489.28,534.28,207.00,526.58,3.50,74.26,Merchant Integrations,5


## 7. Incidents and MTTR
Only resolved incidents can contribute to MTTR. Open records remain visible separately; incident/deployment links are triage associations, not causal evidence.

In [13]:
incident_summary = analysis["incident_team_summary"].sort_values("incident_count", ascending=False)
display(incident_summary.round(2))
display(analysis["incidents_by_team_severity"].round(2))

,team_id,incident_count,open_incidents,resolved_incidents,mean_mttr_hours,p1_incidents,team_name
2,T03,36,1,35,3.64,2,Fulfillment Systems
1,T02,31,2,29,2.33,0,Core Platform
5,T06,28,1,27,3.63,1,SRE & Infrastructure
0,T01,19,0,19,3.43,4,Checkout & Payments
4,T05,12,0,12,3.56,1,Data & Analytics
3,T04,11,2,9,3.95,0,Consumer Mobile
6,T07,11,2,9,3.17,0,Merchant Integrations


,team_id,severity,incident_count,open_incidents,resolved_incidents,mean_mttr_hours,median_mttr_hours,affected_users_estimate,customer_impact_minutes,team_name
0,T01,P1,4,0,4,4.48,4.10,39829,8,Checkout & Payments
1,T01,P2,5,0,5,3.69,3.63,16347,9,Checkout & Payments
2,T01,P3,7,0,7,2.96,3.15,6746,11,Checkout & Payments
3,T01,P4,3,0,3,2.66,2.87,531,5,Checkout & Payments
4,T02,P2,7,0,7,2.99,3.27,22123,12,Core Platform
5,T02,P3,15,2,13,2.56,2.17,9872,22,Core Platform
6,T02,P4,9,0,9,1.46,1.19,1212,10,Core Platform
7,T03,P1,2,0,2,10.63,10.63,14737,9,Fulfillment Systems
8,T03,P2,10,0,10,4.39,3.49,36421,23,Fulfillment Systems
9,T03,P3,19,0,19,2.67,2.88,15772,26,Fulfillment Systems


## 8. Backlog ageing and work-type mix
Backlog ageing uses the final sprint end date in the synthetic dataset as the historical as-of date, not the date you open this notebook.

In [14]:
backlog = analysis["backlog_as_of_dataset_end"]
print("Backlog rows:", len(backlog))
display(backlog[["task_id", "team_name", "task_type", "priority", "status", "age_days_as_of_dataset_end", "blocked_days"]].head(20))
display(analysis["task_type_summary"].round(2))

Backlog rows: 193


,task_id,team_name,task_type,priority,status,age_days_as_of_dataset_end,blocked_days
0,TASK00010,Checkout & Payments,Bug,Medium,Blocked,191,2
29,TASK00144,Core Platform,Spike,High,To Do,188,1
53,TASK00326,Fulfillment Systems,Feature,Medium,Blocked,188,3
52,TASK00316,Fulfillment Systems,Feature,Low,In Progress,186,2
1,TASK00011,Checkout & Payments,Support,Low,In Progress,185,1
145,TASK00731,SRE & Infrastructure,Tech Debt,High,Blocked,185,4
94,TASK00507,Consumer Mobile,Feature,High,In Progress,176,1
54,TASK00336,Fulfillment Systems,Support,Medium,In Progress,174,0
30,TASK00152,Core Platform,Tech Debt,Low,In Progress,172,0
123,TASK00606,Data & Analytics,Tech Debt,High,In Progress,171,2


,team_id,task_type,tasks,done_tasks,mean_cycle_time_days,median_cycle_time_days,mean_story_points,done_pct,team_name
0,T01,Bug,34,26,3.70,2.08,2.85,76.47,Checkout & Payments
1,T01,Feature,65,54,5.28,4.84,3.80,83.08,Checkout & Payments
2,T01,Spike,6,5,2.27,2.42,3.67,83.33,Checkout & Payments
3,T01,Support,20,17,5.99,5.54,3.85,85.00,Checkout & Payments
4,T01,Tech Debt,14,8,3.53,3.48,4.36,57.14,Checkout & Payments
5,T02,Bug,42,35,6.11,6.67,3.45,83.33,Core Platform
6,T02,Feature,67,60,5.95,5.58,4.12,89.55,Core Platform
7,T02,Spike,13,12,3.92,2.16,5.00,92.31,Core Platform
8,T02,Support,18,18,5.44,3.92,3.67,100.00,Core Platform
9,T02,Tech Debt,35,27,4.14,3.67,3.46,77.14,Core Platform


## 9. Translate observations into management questions
Use these tables to identify candidates for investigation. Do not invent root causes from descriptive patterns. For each finding, state the metric, denominator, comparison period, caveat and recommended next diagnostic step.

In [15]:
team_scorecard = (team_tasks[["team_id", "team_name", "headcount", "task_count", "task_completion_pct", "commit_completion_pct", "mean_cycle_time_days"]]
    .merge(deployment_summary[["team_id", "production_deployments", "failure_rate_pct"]], on="team_id", how="left")
    .merge(capacity_summary[["team_id", "utilization_pct"]], on="team_id", how="left"))
team_scorecard.round(2)

,team_id,team_name,headcount,task_count,task_completion_pct,commit_completion_pct,mean_cycle_time_days,production_deployments,failure_rate_pct,utilization_pct
0,T01,Checkout & Payments,8,139,79.14,74.08,4.75,67,10.45,82.45
1,T02,Core Platform,9,175,86.86,83.51,5.44,90,5.56,88.71
2,T03,Fulfillment Systems,10,173,75.72,70.47,5.26,56,7.14,90.12
3,T04,Consumer Mobile,7,110,73.64,74.79,4.46,21,4.76,78.36
4,T05,Data & Analytics,6,129,82.95,82.00,4.51,33,6.06,81.46
5,T06,SRE & Infrastructure,5,104,81.73,82.95,5.00,71,2.82,92.31
6,T07,Merchant Integrations,5,109,73.39,70.95,5.26,21,14.29,74.26


### Questions to take to an engineering manager

1. Are sprint commitments realistic given historical capacity and support work?
2. Which production services have elevated deployment failures, and what do release-level logs show?
3. Which service/environment combinations account for the largest cloud cost or largest budget variance?
4. Does sustained high utilization coincide with ageing work or incident load?
5. Which blocked tasks are oldest, and what dependency or ownership change might unblock them?

**Portfolio conclusion:** summarise two or three data-supported observations, show the underlying denominator, explain uncertainty and recommend the next analysis. Synthetic results are demonstrations of method only; they are not evidence about Zepto or any real company.